# Osaka Geospatial AI — Colab Demo

このNotebookはsetup・script実行・成果物表示だけを行います。GIS・ML・AI処理は `src/` にあります。

初回は配布ZIP `osaka-geospatial-ai.zip` をColabの `/content/` へアップロードするか、下の `REPO_URL` に自分のリポジトリURLを設定してください。GPU推論を使う場合は、ランタイムをGPUに変更して `ENABLE_AI = True` にします。デフォルトはGIS+MLと定型レポートです。

## 1. Environment Setup

In [ ]:
import os
import sys
import subprocess
from pathlib import Path

REPO_URL = ""  # 公開済み/アクセス可能な自分のrepository URL。ZIP利用時は空欄。
ENABLE_AI = False  # Colab GPU + optional AI dependencies
OFFLINE = False  # 全データがキャッシュ済みの場合のみTrue
IN_COLAB = Path("/content").is_dir()
ROOT = Path("/content/osaka-geospatial-ai") if IN_COLAB else (Path.cwd() if Path("pyproject.toml").exists() else Path.cwd().parent)


## 2. Clone / Install

ローカルではREADMEのSetupを先に実行してください。Colabでは以下が依存関係をインストールします。

In [ ]:
if not ROOT.exists():
    if REPO_URL:
        subprocess.run(["git", "clone", REPO_URL, str(ROOT)], check=True)
    elif Path("/content/osaka-geospatial-ai.zip").exists():
        subprocess.run(["unzip", "-q", "/content/osaka-geospatial-ai.zip", "-d", "/content"], check=True)
    else:
        raise FileNotFoundError("配布ZIPを/contentへアップロードするかREPO_URLを設定してください。")
os.chdir(ROOT)
if IN_COLAB:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", ".[ai]" if ENABLE_AI else "."], check=True)
from IPython.display import display, Image, HTML, JSON, Markdown
from osaka_geo_ai.presentation import dataset_summary, metrics_table, prediction_preview


## 3. Configuration

標準設定は2025年までの地価を使用します。AI有効時は4枚のPNGをVLMへ、analysis.jsonをLLMへ渡します。モデル名・リビジョン・メモリ上限はconfigs/llm.yamlにあります。

In [ ]:
import yaml
CONFIG = ROOT / "configs/colab_runtime.yaml"
CONFIG.write_text(yaml.safe_dump({"includes": ["default.yaml"], "vlm": {"enabled": ENABLE_AI}, "llm": {"enabled": ENABLE_AI}}, sort_keys=False), encoding="utf-8")
display(Markdown(f"Config: `{CONFIG.name}` / AI enabled: **{ENABLE_AI}**"))


## 4. Run Pipeline

進捗はscriptのloggingで表示されます。初回は国土交通省の公式ZIPを取得します。

In [ ]:
command = [sys.executable, "scripts/run_pipeline.py", "--config", str(CONFIG)]
if OFFLINE:
    command.append("--offline")
subprocess.run(command, check=True)


## 5. Dataset Summary

人口は2010年国勢調査を基準にした古い将来推計です。現人口ではありません。

In [ ]:
display(dataset_summary(ROOT))

## 6. Interactive GIS Map

探索用HTMLです。地図タイルは使いません。FoliumのJavaScript/CSS読込にはインターネット接続が必要です。

In [ ]:
display(HTML(filename=str(ROOT / "artifacts/maps/land_price_map.html")))

## 7. Static Maps

観測値・予測・残差を区別してください。正の残差は過小予測、負は過大予測です。4面比較の人口・鉄道の参照年は地価図と異なります。

In [ ]:
display(Image(filename=str(ROOT / "artifacts/figures/land_price_map.png"), width=650))
display(Image(filename=str(ROOT / "artifacts/figures/prediction_map.png"), width=650))
display(Image(filename=str(ROOT / "artifacts/figures/vlm_input_overview.png"), width=1000))


## 8. Model Metrics

選択はvalidation MAEだけを用います。mae/rmseは変化率fraction、mae_pp/rmse_ppはパーセントポイントです。

In [ ]:
display(metrics_table(ROOT))

## 9. Prediction Results

ここでは2025年のテスト予測を表示します。最新設定年からの別予測はartifacts/predictions/forecast.parquetです。

In [ ]:
display(prediction_preview(ROOT))

## 10. VLM Analysis

completed以外の場合、モデルによる視覚的観察はありません。JSONのstatusとlimitationsを確認してください。

In [ ]:
display(JSON(filename=str(ROOT / "artifacts/vlm/map_analysis.json")))

## 11. Final Report

AI無効時は数値から作る定型説明です。有効時のLLM説明は本文とは別に、内容未検証の生成文章として追記されます。

In [ ]:
display(JSON(filename=str(ROOT / "artifacts/reports/report_generation.json")))
display(Markdown((ROOT / "artifacts/reports/report.md").read_text(encoding="utf-8")))
